# 07B – SHAP Local Explainability

Enterprise notebook for explaining **individual bankruptcy predictions**.

## Business Objective

Explain why the model predicted a specific company as high or low bankruptcy risk using local SHAP explanations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
import joblib
from sklearn.model_selection import train_test_split

In [ ]:
MODEL_PATH='production_bankruptcy_model.joblib'
DATA_PATH='american_bankruptcy.csv'
INSTANCE_INDEX=0

model=joblib.load(MODEL_PATH)
df=pd.read_csv(DATA_PATH)

if 'status_label' in df.columns:
    y=df['status_label'].map({'alive':0,'failed':1})
    X=df.drop(columns=[c for c in ['status_label','company_name'] if c in df.columns])
elif 'target' in df.columns:
    y=df['target']
    X=df.drop(columns=[c for c in ['target','company_name'] if c in df.columns])
else:
    raise ValueError('Target column not found')

_,X_test,_,_=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
instance=X_test.iloc[[INSTANCE_INDEX]]

In [ ]:
predict_fn=lambda d:model.predict_proba(d)[:,1] if hasattr(model,'predict_proba') else model.predict(d)

background=X_test.sample(min(50,len(X_test)),random_state=42)
explainer=shap.Explainer(predict_fn,background)
shap_values=explainer(instance)

In [ ]:
# Waterfall Plot
shap.plots.waterfall(shap_values[0],max_display=15,show=False)
plt.tight_layout()
plt.savefig('local_waterfall.png',dpi=300)
plt.show()

In [ ]:
# Force Plot
force=shap.plots.force(
    shap_values.base_values[0],
    shap_values.values[0],
    instance.iloc[0],
    matplotlib=True,
    show=False
)
plt.savefig('local_force_plot.png',dpi=300,bbox_inches='tight')
plt.show()

In [ ]:
# Local Feature Contributions
local=pd.DataFrame({
    'Feature':instance.columns,
    'Feature_Value':instance.iloc[0].values,
    'SHAP_Value':shap_values.values[0]
}).sort_values('SHAP_Value',key=np.abs,ascending=False)

local.to_csv('local_shap_explanation.csv',index=False)
local.head(20)

## Business Interpretation

- Positive SHAP values increase bankruptcy risk for this company.
- Negative SHAP values decrease bankruptcy risk.
- The largest absolute SHAP values identify the primary reasons behind this prediction.
- Analysts can use this explanation to justify model decisions and communicate risk clearly.

## Deliverables

- `local_waterfall.png`
- `local_force_plot.png`
- `local_shap_explanation.csv`

This notebook provides local explainability for individual predictions and complements the global SHAP analysis from 07A.